In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Load and preprocess historical IPO data
data = pd.read_csv('/content/IPO Data.csv')
data['IPO_Size'] = data['IPO_Size'].str.replace(r'[₹,]', '', regex=True)
data['IPO_Size'] = data['IPO_Size'].str.replace(r'[Cc]r', '', regex=True).str.strip()
data['IPO_Size'] = data['IPO_Size'].astype(float) * 1e7
data['Subscription'] = data['Subscription'].str.replace('x', '', regex=False).astype(float)
data['IPO Price'] = data['IPO Price'].str.replace(r'[^0-9.]', '', regex=True).astype(float)
data['Listing Price'] = data['Listing Price'].str.extract(r'@(\d+\.\d+)').astype(float)
data['GMP'] = data['GMP'].str.replace(r'₹', '', regex=True).astype(float)
data['apply_for_ipo'] = (data['Listing Price'] > data['IPO Price'] * 1.20).astype(int)
data = data.dropna(subset=['IPO Price', 'Listing Price', 'Subscription', 'GMP', 'IPO_Size'])

# Feature engineering
data['Subscription_tier'] = pd.qcut(data['Subscription'], q=3, labels=['low', 'medium', 'high'])
data['GMP_to_IPO_Ratio'] = data['GMP'] / data['IPO Price']
data['Log_IPO_Size'] = np.log1p(data['IPO_Size'])
data['Subscription_to_GMP'] = data['Subscription'] * data['GMP']
data['IPO_Price_to_Size_Ratio'] = data['IPO Price'] / data['IPO_Size']

X = data[['IPO_Size', 'Subscription', 'GMP', 'IPO Price', 'GMP_to_IPO_Ratio', 'Log_IPO_Size', 'Subscription_to_GMP', 'IPO_Price_to_Size_Ratio']]
y = data['apply_for_ipo']

# Train/test split and Random Forest model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# Feature importance
importances = model.feature_importances_
for i, col in enumerate(X.columns):
    print(f'{col}: {importances[i]}')

# Visualizations
plt.figure(figsize=(8, 5))
sns.countplot(x='Subscription_tier', data=data, palette='viridis')
plt.title('Distribution of Subscription Tiers')
plt.xlabel('Subscription Tier')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(x=data['GMP_to_IPO_Ratio'], y=y, hue=y, palette='cool', s=50)
plt.title('GMP-to-IPO Price Ratio vs Apply Decision')
plt.xlabel('GMP-to-IPO Price Ratio')
plt.ylabel('Apply for IPO')
plt.legend([],[], frameon=False)
plt.show()

plt.figure(figsize=(8, 6))
sns.histplot(data['Log_IPO_Size'], kde=True, color='lightblue', bins=30)
plt.title('Distribution of Log IPO Size')
plt.xlabel('Log IPO Size')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(10, 8))
correlation_matrix = data[['IPO_Size', 'Subscription', 'GMP', 'IPO Price', 'GMP_to_IPO_Ratio', 'Log_IPO_Size', 'Subscription_to_GMP', 'IPO_Price_to_Size_Ratio']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='Blues', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Load upcoming IPO data
new_data = pd.read_excel('/content/Mainline Upcoming IPO.xlsx')
new_data['Subscription'] = new_data['Subscription'].str.replace('x', '', regex=False).astype(float)

required_cols = ['IPO_Size', 'Subscription', 'GMP', 'IPO Price']
new_data = new_data.dropna(subset=required_cols)

# Apply the same feature engineering used during training
new_data['GMP_to_IPO_Ratio'] = new_data['GMP'] / new_data['IPO Price']
new_data['Log_IPO_Size'] = np.log1p(new_data['IPO_Size'])
new_data['Subscription_to_GMP'] = new_data['Subscription'] * new_data['GMP']
new_data['IPO_Price_to_Size_Ratio'] = new_data['IPO Price'] / new_data['IPO_Size']

X_new = new_data[['IPO_Size', 'Subscription', 'GMP', 'IPO Price', 'GMP_to_IPO_Ratio', 'Log_IPO_Size', 'Subscription_to_GMP', 'IPO_Price_to_Size_Ratio']]

# Generate probability-based recommendations
new_data['success_probability'] = model.predict_proba(X_new)[:, 1]
new_data['recommend_to_apply'] = (new_data['success_probability'] >= 0.5).astype(int)
new_data[['IPO', 'success_probability', 'recommend_to_apply']]

In [ ]:
new_data.to_excel('/content/IPO_Prediction_Output.xlsx', index=False)